# 35장 · RAG 평가 데이터셋 구성

평가 문항에는 질문 외에 기대하는 근거와 답변 가능 여부가 필요합니다. 결과를 본 뒤 기준을 바꾸지 않도록 개발용과 확인용 문항을 분리합니다.

평가 질문마다 기대 문서 ID와 답변 가능 여부를 먼저 기록합니다. 질문·자료·판정 기준을 고정해야 다음 판에서 같은 실패가 해소됐는지 비교할 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 질문 ID와 기대 근거 ID

question_id는 다시 시험할 질문을 찾는 이름이고 expected_document_ids는 답이 실린 원문을 찾는 이름입니다. 페이지가 여러 청크로 갈라져도 기대 근거는 원문에 연결합니다.


### 셀 01 · 질문과 기대 근거 ID 매핑

가상 도서관 공통 자료의 첫 개발 문항과 연결된 원문을 읽으세요.


In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
item = {'question_id': 'dev-01',
 'question': '대출 기간은 며칠인가요?',
 'expected_document_ids': ['library-1'],
 'answerable': True,
 'split': 'development',
 'source_split': 'development'}
by_id = {doc['id']: doc for doc in records}
print(by_id[item['expected_document_ids'][0]]['text'])

도서 대출 기간은 14일입니다.


library-1에서 14일이라는 사실을 읽었습니다. 기대 답의 표현이 달라도 이 숫자와 대출이라는 조건이 유지되는지 볼 수 있습니다.

평가 문항에 기대 답 ‘14일’만 남기면 모델이 엉뚱한 문단을 보고 같은 숫자를 말해도 통과할 수 있습니다. 어떤 문서에 그 기간이 적혀 있는지 ID도 함께 적어 두면 검색이 맞았는지 먼저 확인할 수 있습니다.


### 대표 문항의 적용 범위

<p>대표 질문은 우리가 확인하려는 사용 상황을 드러내는 질문입니다. 여기서는 대출 기간, 휴관 요일, 반납함 이용 시간만 원문에 있습니다. 따라서 이 세 주제와 원문에 없는 질문을 나누어 생각합니다. 질문 수를 늘리는 것보다 어떤 상황을 포함하고 어떤 상황을 아직 다루지 않는지 먼저 기록합니다.</p>

<table><caption>현재 자료의 질문 범위</caption><thead><tr><th scope="col">상황</th><th scope="col">질문 예</th><th scope="col">확인 근거</th><th scope="col">범위 밖</th></tr>
</thead><tbody><tr><td>명시된 규칙</td><td>대출 기간은 며칠인가요?</td><td>library-1의 14일</td><td>연장·연체 규칙</td></tr>
<tr><td>일정</td><td>휴관 요일은 언제인가요?</td><td>library-2의 월요일</td><td>공휴일 운영</td></tr>
<tr><td>이용 가능 시간</td><td>반납함은 몇 시간 이용할 수 있나요?</td><td>library-3의 24시간</td><td>위치·접근 방법</td></tr>
<tr><td>자료에 없는 정보</td><td>주차 요금은 얼마인가요?</td><td>없음</td><td>실제 기관 요금 추측</td></tr>
</tbody></table>


### 문항이 속한 세트 기록

<p>question_id는 문항을 다시 불러올 때 사용하고 expected_document_ids는 정답 근거를 가리킵니다. split은 개발 중 반복해 볼 문항과 설정을 확정한 뒤 볼 문항을 나눕니다. 문구만 바꾼 질문을 두 세트에 복사하면 새로운 표현을 얼마나 잘 처리하는지 평가하기 어렵습니다.</p>


### 셀 02 · 근거 없는 평가 문항

주차 요금 문항의 기대 문서가 왜 비어 있는지 원문 세 쪽과 대조하세요.


In [2]:
parking = {'question_id': 'dev-03',
 'question': '주차 요금은 얼마인가요?',
 'expected_document_ids': [],
 'answerable': False,
 'split': 'development',
 'source_split': 'development'}
print(parking['answerable'], parking['expected_document_ids'])

False []


False와 빈 목록은 답을 유보해야 한다는 뜻입니다. 평가 자료를 덜 채워 둔 경우와 구분하려고 answerable을 함께 기록합니다.


### 근거가 없는 문항의 기대값

<p>주차 요금 질문은 문장이 명확하지만 원문에 답이 없습니다. 파이썬 문항에는 answerable을 False로, expected_document_ids를 빈 목록으로 적습니다. 모르는 가격을 0원이나 무료로 채우면 자료에 없는 정답을 만든 셈이 됩니다.</p>

<p>“언제 이용할 수 있나요?”처럼 대상이 불명확한 질문은 다른 문제입니다. 도서관 개관 시간인지 반납함 시간인지 먼저 정해야 판정자가 같은 기준으로 읽을 수 있습니다. 이 기본 세트에는 모호한 질문을 억지로 답변 가능/불가에 넣기보다 문장을 고치고 수정 이유를 기록합니다.</p>

<p>검색 서버의 timeout은 질문의 답변 가능 여부를 바꾸지 않습니다. 원문에 14일이 있는 질문은 서버가 실패해도 답변 가능한 질문입니다. 운영 실패는 36장의 실행 기록에서 분리합니다.</p>

<table><caption>세 상황의 처리</caption><thead><tr><th scope="col">상황</th><th scope="col">질문 세트에서 할 일</th><th scope="col">실행 기록에서 할 일</th></tr>
</thead><tbody><tr><td>원문에 답 없음</td><td>false와 빈 기대 목록</td><td>정상 무응답 여부 확인</td></tr>
<tr><td>질문 대상 모호</td><td>질문을 명확히 하거나 별도 정책 명시</td><td>기준 확정 전 점수 보류</td></tr>
<tr><td>검색 서버 오류</td><td>기존 기대 근거 유지</td><td>retrieval 오류로 기록</td></tr>
</tbody></table>


### 답변을 판정할 때 확인할 세 가지

<p>대출 질문이라면 검색 결과에 library-1이 있었는지, 답변이 그 ID를 가리키는지, 기간을 14일로 옮겼는지를 각각 확인합니다. 앞의 두 항목이 맞아도 마지막 숫자가 틀릴 수 있습니다.</p>


## 개발용·확인용 평가 문항 분리

개발 문항으로는 검색 설정을 고칩니다. 이후 확인할 질문까지 보며 여러 설정을 고르면 그 질문이 새 표현에서의 성능을 보여 주기 어렵습니다. 책에 공개된 확인 문항은 팀이 보지 못한 비공개 문항과 구분해 기록합니다.


### 여러 문항을 JSONL로 저장할 때

<p>노트북의 item 한 건은 딕셔너리입니다. 문항을 파일로 모을 때는 JSONL 한 줄에 한 문항을 저장할 수 있습니다. 그러면 각 줄을 읽어 같은 ID 검사와 원문 대조를 반복할 수 있습니다.</p>


### 과제 03 · 개발 문항의 기대값 작성

dev-02에 ‘휴관 요일은 언제인가요?’와 기대 원문 ID, 응답 가능 여부를 넣어 문서를 찾아보세요.

이 연습에서는 휴관 질문의 핵심 필드를 작성합니다. 평가 파일로 보관할 때는 개발에 사용한 질문임을 나타내는 split 값 development도 같은 항목에 기록하세요. 확인용 문항과 섞이지 않게 해 줍니다.


In [3]:
# dev-02의 질문, 기대 문서 ID, answerable을 채우세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>holiday = {&#x27;question_id&#x27;:&#x27;dev-02&#x27;, &#x27;question&#x27;:&#x27;휴관 요일은 언제인가요?&#x27;, &#x27;expected_document_ids&#x27;:[&#x27;library-2&#x27;], &#x27;answerable&#x27;: True, &#x27;split&#x27;: &#x27;development&#x27;, &#x27;source_split&#x27;: &#x27;development&#x27;}
print(by_id[holiday[&#x27;expected_document_ids&#x27;][0]][&#x27;text&#x27;])</code></pre>
<p>실행 결과</p><pre><code>도서관은 월요일에 휴관합니다.</code></pre>
<p>dev-02는 개발 중 다시 볼 수 있는 문항입니다. 기대 문서 library-2를 열면 2쪽에서 월요일 휴관 문장을 찾을 수 있습니다.</p>
</details>


### 설정을 고르는 질문과 확인하는 질문

<p>개발 문항은 검색 규칙을 바꿀 때 반복해서 사용합니다. 확인 문항은 설정을 정한 뒤 다른 질문에서도 작동하는지 살피는 데 씁니다. 확인 결과를 보고 규칙을 다시 바꿨다면 어떤 문항을 참고했는지 기록하세요.</p>


### 과제 04 · 확인용 문항의 근거 작성

“무인 반납함을 밤에도 쓸 수 있나요?”를 개발 세트와 분리해 public_check 문항으로 작성하세요. 기대 문서 ID로 원문을 조회해 밤에도 이용할 수 있다는 근거를 읽으세요.


In [ ]:
question = '무인 반납함을 밤에도 쓸 수 있나요?'
# 질문, 기대 근거, 답변 가능 여부, 분할 이름을 한 레코드에 보관하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>check = {
    &#x27;question_id&#x27;: &#x27;check-return&#x27;, &#x27;question&#x27;: &#x27;무인 반납함을 밤에도 쓸 수 있나요?&#x27;,
    &#x27;expected_document_ids&#x27;: [&#x27;library-3&#x27;], &#x27;answerable&#x27;: True,
    &#x27;split&#x27;: &#x27;public_check&#x27;,
}
print(check[&#x27;split&#x27;], by_id[check[&#x27;expected_document_ids&#x27;][0]][&#x27;text&#x27;])</code></pre>
<p>실행 결과</p><pre><code>public_check 무인 반납함은 24시간 이용할 수 있습니다.</code></pre>
<p>24시간이라고 쓰인 대상은 무인 반납함입니다. “도서관도 24시간 개방한다”는 답은 같은 숫자를 인용해도 범위를 잘못 옮긴 답입니다. 이 문항은 풀이까지 공개했으므로 모델을 최종 평가할 비공개 질문으로 다시 세지 않습니다.</p>
</details>


## 돌아보기

이번에는 상담 운영 시간을 묻는 질문을 만드세요. 현재 자료로 답할 수 있는지 확인한 뒤 기대할 응답을 적어 두세요.
